# World Cup Transit Service near SoFi Stadium

In [ ]:
import warnings
warnings.filterwarnings("ignore")
                        
import altair as alt
import branca.colormap as cm
import folium
import geopandas as gpd
import google.auth
import pandas as pd

import world_cup_vars as wc_vars
import D1_prep_trips as D1
import D2_prep_stop_arrivals as D2
import chart_utils
import _color_palette

credentials, _ = google.auth.default()

## Regional Trips

In [ ]:
sofi_trips = D1.filter_fct_daily_schedule_rt_route_direction_summary_to_special_routes(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.socal_names,
    route_name_dict = wc_vars.special_socal_routes_dict,
    event_time_of_day_dict = wc_vars.sofi_match_times  
)

In [ ]:
daily_trips_by_operator = D1.aggregate_daily_trips(
    sofi_trips, ["service_date", "schedule_name"])

In [ ]:
# just add the before and after, set to zero, otherwise LA Metro Events 
# single point drops away from chart visibility
append_la_metro_events = pd.DataFrame({
    'service_date': ["2026-07-09", "2026-07-11"],
})
append_la_metro_events = append_la_metro_events.assign(
    service_date = pd.to_datetime(append_la_metro_events.service_date).dt.normalize(),
    schedule_name = "LA Metro Events Schedule",
    n_trips = 0
)

daily_trips_by_operator2 = pd.concat(
    [daily_trips_by_operator, append_la_metro_events], 
    axis=0, ignore_index=True
)

In [ ]:
chart_utils.trip_chart_with_event_dates(
    daily_trips_by_operator2, wc_vars.sofi_dates, color_col="schedule_name"
).properties(
    title= "Daily Trips by Operator during World Cup",
    width=500, height=300
)

## Trips by Route

In [ ]:
daily_trips_by_route = D1.aggregate_daily_trips(
    sofi_trips, ["service_date", "schedule_name", "route_name"]
)

In [ ]:
chart_utils.trip_chart_with_event_dates(
    daily_trips_by_route, wc_vars.sofi_dates, color_col="route_name"
).properties(
    title= "Daily Trips by Route during World Cup",
    width=500, height=300
)

In [ ]:
# Get each route's daily average number of trips on event days vs non-event days - plot on map
trips_by_event = D2.aggregate_by_event_type(
    sofi_trips, group_cols = ["schedule_name", "route_name", "event_day", "day_type"], 
    metric_cols = ["n_trips"]
).rename(columns = {
    "n_trips": "daily_trips", 
})

trips_wide = D2.make_wide(
    trips_by_event,
    index_cols=["schedule_name", "route_name"],
    pivot_cols=["day_type", "event_day"],
    value_cols=["daily_trips"],
)

trips_wide = trips_wide.assign(
    change_daily_trips = trips_wide[["change_daily_trips_weekday", "change_daily_trips_weekend"]].sum(axis=1),          
)

# Add route's line geometry
route_change_gdf = pd.merge(
    sofi_trips[["schedule_name", "route_name", "geometry"]].drop_duplicates(),
    trips_wide,
    on = ["schedule_name", "route_name"],
    how = "inner"
)

In [ ]:
poi = gpd.read_parquet(
    f"{wc_vars.GCS_FILE_PATH}points_of_interest_{wc_vars.event_name}.parquet",
    storage_options = {"token": credentials.token},
    filters = [[("point_of_interest", "==", "SoFi Stadium")]]
)

In [ ]:
# check that our change column works with the shared legend cutoff across the maps in this notebook
route_change_gdf.change_daily_trips.min(), route_change_gdf.change_daily_trips.max()

In [ ]:
# check that our change column works with the shared legend cutoff across the maps in this notebook
arrivals_wide.combined_change_daily_arrivals.min(), arrivals_wide.combined_change_daily_arrivals.max()

In [ ]:
SERVICE_CHANGE_COLORS = [
    "#EE6363", #indianred2
    # pick ones from YlGnBu palette and reverse so yellow is most
    #"#081D58", "#225EA8", "#41B6C4", "#C7E9B4", "#FFFF00",
    # pick ones from BuGn palette, but set closer to zero as a light yellow
    "#d3d3d3", # how to single out the zeroes
    "#C7E9B4", "#66C2A4", "#41AE76", "#006D2C", "#00441B",
]
SERVICE_CHANGE_INDEX = [-25, 0, 1, 25, 50, 100, 150, 200]
SERVICE_CHANGE_CAPTION = "service change (positive = added service; negative = reduced service)"

In [ ]:
# need to do fillna
arrivals_wide[arrivals_wide.combined_change_daily_arrivals.isna()]

In [ ]:
route_map = route_change_gdf.explore(
    "change_daily_trips",
    tiles = "CartoDB Positron",
    cmap = cm.StepColormap(
        colors=SERVICE_CHANGE_COLORS, 
        index=SERVICE_CHANGE_INDEX, 
        vmin=-25, vmax=200, # this is manually here after a check that max is 78
        tick_labels=SERVICE_CHANGE_INDEX,
        caption=SERVICE_CHANGE_CAPTION
    ),
    name = "Additional Trips (compared to baseline)" 
)

route_map = arrivals_wide.explore(
    "combined_change_daily_arrivals",
    m = route_map,
    cmap = cm.StepColormap(
        colors=SERVICE_CHANGE_COLORS, 
        index=SERVICE_CHANGE_INDEX, 
        vmin=-25, vmax=200, # this is manually here after a check that max is 164
        tick_labels=SERVICE_CHANGE_INDEX,
        caption=SERVICE_CHANGE_CAPTION
    ),
    name = "Additional Arrivals (compared to baseline)" 
)

# https://python-visualization.github.io/folium/latest/user_guide/geojson/geojson_marker.html
# available colors, https://python-visualization.github.io/folium/latest/reference.html#folium.map.Marker
route_map = poi.explore(
    "point_of_interest",
    m=route_map,
    marker_type = "marker",
    name="SoFi Stadium",
    legend = False, # legend color is blue no matter what color is set for icon, confusing
    marker_kwds=dict(icon=folium.Icon(icon="star", color="darkpurple")),
)

folium.LayerControl().add_to(route_map)
route_map

In [ ]:
arrivals_wide.combined_change_daily_arrivals.value_counts()

## LA Metro World Cup Feed
Thicker routes have more trips!

In [ ]:
la_metro_events = sofi_trips[sofi_trips.schedule_name == "LA Metro Events Schedule"].reset_index(drop=True)

In [ ]:
#https://matplotlib.org/stable/users/explain/colors/colormaps.html
la_metro_events[["route_name", "direction_id", "geometry", "n_trips"]].explore(
    "route_name",
    tiles = "CartoDB Positron",
    style_kwds={"style_function": lambda x: {"weight":x["properties"]["n_trips"]*0.05}},
    cmap="Set1"
)

## Stop Arrivals

In [ ]:
sofi_stop_arrivals = D2.filter_fct_daily_scheduled_stops_to_special_routes(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.socal_names,
    route_name_dict = wc_vars.special_socal_routes_dict,
    event_time_of_day_dict = wc_vars.sofi_match_times
)

arrivals_wide = D2.stop_arrival_change_from_baseline_wide(sofi_stop_arrivals)

In [ ]:
operator_df = (
    arrivals_wide
    .groupby(["schedule_name", "route_id_array", "stop_name"])
    .agg({
        "change_daily_arrivals_weekday": "sum",
        "change_daily_arrivals_weekend": "sum",
        "stop_id": "nunique"
    })
    .reset_index()
    .rename(columns = {"stop_id": "n_stop_ids"})
)

In [ ]:
for i in sorted(operator_df.schedule_name.unique()):
    chart = chart_utils.weekday_weekend_chart_by_operator(operator_df, i)
    display(chart)